In [1]:
import os
import pandas as pd
import numpy as np
import openpyxl
import xlsxwriter as xlwt
import xlrd
from openpyxl import Workbook
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import glob

In [57]:
os.chdir(r'G:\Work CIC\Work CIC\02 Control\07 Spring 2023-2024\AA')

In [58]:
Reg= pd.read_excel(r'G:\Work CIC\Work CIC\02 Control\07 Spring 2023-2024\Control Sheets\RegList6-2_V1.xlsx')
Grade= pd.read_excel(r'G:\Work CIC\Work CIC\02 Control\07 Spring 2023-2024\Control Sheets\CoursesFall2021.xlsx')
#FALL2021_Grades with Vlookup

In [59]:
Reg.head()

,CIC ID,Name,Program,Concentration,Course Code,Course Name,CW,FE,PG,RETAKE,Attend,PGCheck
0,201307092,Philopateer Essam Waheeb Abdelqodous,EGYPT,E-CIVL,CIS 451,Foundations Engineering 2,38,30.0,0,0,Attend,0
1,201307092,Philopateer Essam Waheeb Abdelqodous,EGYPT,E-CIVL,BAS 211,Mathematics 4,13,10.0,0,1,Attend,0
2,201307092,Philopateer Essam Waheeb Abdelqodous,EGYPT,E-CIVL,CIS 321,Design of Concrete Structures 2,28,22.0,0,1,Attend,0
3,201406034,Mohamed Tarek Abdelfatah Abdelhady,EGYPT,E-CIVL,BAS 211,Mathematics 4,0,0.0,0,1,Abs.,0
4,201406034,Mohamed Tarek Abdelfatah Abdelhady,EGYPT,E-CIVL,HUM 111,Technical Report Writing,0,0.0,0,1,Abs.,0


In [60]:
Reg.sort_values("CIC ID", inplace=True)
Reg['CIC ID'].astype(int)
Reg['CW'].replace('',5000, inplace=True)
#Reg['CW']=Reg['CW'].astype(int)

In [61]:
CourseCode= Reg['Course Code'].unique().tolist()
Reg['CourseName']= Reg['Course Code']+'_'+Reg['Course Name']
LA= Reg['CourseName'].unique().tolist()

In [62]:
CS = dict()
for k, v in Reg.groupby('Course Code'):
    CS[k] = v

In [63]:
# Don't Forget to Modifiy The Detailed Registration
len(LA)

124

In [64]:
len(CourseCode)

124

In [65]:
df= pd.DataFrame()

In [66]:
df["LA"]=LA

df.to_csv(r'G:\Work CIC\Work CIC\02 Control\07 Spring 2023-2024\Control Sheets\Coursenames.csv')

In [67]:
df2= pd.DataFrame()
df2['CourseCode']=CourseCode
df2.to_csv(r'G:\Work CIC\Work CIC\02 Control\07 Spring 2023-2024\Control Sheets\Courscode.csv')

In [68]:
template_file= r'G:\Work CIC\Work CIC\02 Control\07 Spring 2023-2024\Control Sheets\TT.xlsx'
for i in range(0, len(CourseCode)):
    filename= LA[i]
    writer= pd.ExcelWriter
    Column_names=['Name','CIC ID','CW','FE']
    df= CS['{}'.format(CourseCode[i])].drop(['Program','Course Code','Course Name','CourseName'], axis=1).reindex(columns= Column_names)
    wb= load_workbook(template_file)   
    ws= wb['2012 (new)']
    rows= dataframe_to_rows(df.loc[df['CIC ID']<201700000], index= False, header= None)
    for r_idx, row in enumerate(rows,1):
        for c_idx, value in enumerate(row,1):
            ws.cell(row=r_idx+6,column= c_idx+2, value= value)
    ws= wb['2017 (new)'] 
    rows= dataframe_to_rows(df.loc[(df['CIC ID']>201700000)], index= False, header= None)
    for r_idx, row in enumerate(rows,1):
        for c_idx, value in enumerate(row,1):
            ws.cell(row=r_idx+6,column= c_idx+2, value= value)
    A= Grade['CWO'][Grade['code']=='{}'.format(CourseCode[i])].to_numpy()[0]
    B= Grade['FEO'][Grade['code']=='{}'.format(CourseCode[i])].to_numpy()[0]
    C= Grade['CWN'][Grade['code']=='{}'.format(CourseCode[i])].to_numpy()[0]
    D= Grade['FEN'][Grade['code']=='{}'.format(CourseCode[i])].to_numpy()[0]
    ws= wb['2012 (new)']
    ws.cell(row=3,column= 8, value= A)
    ws.cell(row=3,column= 9, value= B)
    ws= wb['2017 (new)']
    ws.cell(row=3,column= 8, value= C)
    ws.cell(row=3,column= 9, value= D)
    CN=['CIC ID','RETAKE','Attend','PG','Concentration','Program']
    df1= CS['{}'.format(CourseCode[i])].reindex(columns= CN)
    ws= wb['2012 (new)']
    rows= dataframe_to_rows(df1.loc[df1['CIC ID']<201700000],index= False, header= None)
    for r_idx, row in enumerate(rows,1):
        for c_idx, value in enumerate(row,1):
            ws.cell(row=r_idx+6,column= c_idx+17, value= value)
    ws= wb['2017 (new)']
    rows= dataframe_to_rows(df1.loc[df1['CIC ID']>201700000],index= False, header= None)
    for r_idx, row in enumerate(rows,1):
        for c_idx, value in enumerate(row,1):
            ws.cell(row=r_idx+6,column= c_idx+17, value= value)
    #max_row= len(ws['C'])
    #ws.print_area= "A1:E{}".format(max_row)
    wb.save(filename+'.xlsx')